# Cuantificadores resueltos con tablas de verdad

Cuaderno complementario al capítulo *2.2 Cuantificador Universal, Existencial y de Unicidad*.

**Idea central.** Sobre un universo **finito** $U=\{u_1,\dots,u_n\}$ los cuantificadores son abreviaturas de conectivos:

$$
(\forall x)P(x)\equiv P(u_1)\land\cdots\land P(u_n),\qquad
(\exists x)P(x)\equiv P(u_1)\lor\cdots\lor P(u_n)
$$

Por eso se pueden evaluar con una tabla de verdad: una fila por elemento del universo y una columna final que reduce la columna de $P(x)$ con $\land$ (∀) o $\lor$ (∃).

Contenido:
1. Cuantificador universal ∀
2. Cuantificador existencial ∃
3. Existencia única ∃!
4. Negación de cuantificadores (De Morgan generalizadas)
5. Cuantificadores múltiples y orden
6. Definiciones tipo ε–δ en un modelo finito

In [1]:
import itertools
import pandas as pd

def V(b):
    # Muestra booleanos como V/F
    return 'V' if b else 'F'

def tabla(U, P, nombre='P(x)'):
    # Tabla de verdad de un predicado unario sobre un universo finito.
    df = pd.DataFrame({'x': list(U), nombre: [V(P(x)) for x in U]})
    return df.set_index('x')

def universal(U, P):   return all(P(x) for x in U)
def existencial(U, P): return any(P(x) for x in U)
def unico(U, P):       return sum(1 for x in U if P(x)) == 1

def resumen(U, P, nombre):
    t = [x for x in U if P(x)]
    return pd.Series({
        'T_P (conjunto de verdad)': set(t),
        '∀x P(x)': V(universal(U, P)),
        '∃x P(x)': V(existencial(U, P)),
        '∃!x P(x)': V(unico(U, P)),
        'contraejemplos': [x for x in U if not P(x)],
    }, name=nombre)

## 1. Cuantificador universal $(\forall x\in U)\,P(x)$

**Regla de verdad:** $\forall x\,P(x)$ es V ⇔ $T_P=U$ (toda fila de $P$ es V). Es F si hay un **contraejemplo**.

Universo $U=\{1,2,3,4,5\}$ y tres predicados:
- $P(x)$: $x>0$
- $Q(x)$: $x<4$
- $R(x)$: $x>5$

In [2]:
U = [1, 2, 3, 4, 5]
P = lambda x: x > 0
Q = lambda x: x < 4
R = lambda x: x > 5

t = pd.concat([tabla(U, P, 'P(x): x>0'), tabla(U, Q, 'Q(x): x<4'), tabla(U, R, 'R(x): x>5')], axis=1)
t.loc['∀x (conjunción de la columna)'] = [V(universal(U, f)) for f in (P, Q, R)]
t

,P(x): x>0,Q(x): x<4,R(x): x>5
x,,,
1,V,V,F
2,V,V,F
3,V,V,F
4,V,F,F
5,V,F,F
∀x (conjunción de la columna),V,F,F


**Lectura.**
- $\forall x\,(x>0)$: la columna es toda V → **V** ($T_P=U$).
- $\forall x\,(x<4)$: aparecen F en $x=4,5$ → **F**; los contraejemplos son $4$ y $5$.
- $\forall x\,(x>5)$: columna toda F → **F** (cualquier elemento es contraejemplo).

In [3]:
pd.DataFrame([resumen(U, P, 'x>0'), resumen(U, Q, 'x<4'), resumen(U, R, 'x>5')])

,T_P (conjunto de verdad),∀x P(x),∃x P(x),∃!x P(x),contraejemplos
x>0,"{1, 2, 3, 4, 5}",V,V,F,[]
x<4,"{1, 2, 3}",F,V,F,"[4, 5]"
x>5,{},F,F,F,"[1, 2, 3, 4, 5]"


## 2. Cuantificador existencial $(\exists x\in U)\,P(x)$

**Regla de verdad:** $\exists x\,P(x)$ es V ⇔ $T_P\neq\emptyset$ (basta **una** V en la columna; ese elemento es el *testigo*). Es F ⇔ la columna es toda F.

Mismo universo y predicados; la columna final reduce con $\lor$.

In [4]:
t = pd.concat([tabla(U, P, 'P(x): x>0'), tabla(U, Q, 'Q(x): x<4'), tabla(U, R, 'R(x): x>5')], axis=1)
t.loc['∃x (disyunción de la columna)'] = [V(existencial(U, f)) for f in (P, Q, R)]
t

,P(x): x>0,Q(x): x<4,R(x): x>5
x,,,
1,V,V,F
2,V,V,F
3,V,V,F
4,V,F,F
5,V,F,F
∃x (disyunción de la columna),V,V,F


- $\exists x\,(x>0)$ → **V** (testigos: todos).
- $\exists x\,(x<4)$ → **V** (testigos: $1,2,3$).
- $\exists x\,(x>5)$ → **F** ($T_R=\emptyset$).

Observación de la tabla: $\forall x\,P(x)\Rightarrow\exists x\,P(x)$ **siempre que $U\neq\emptyset$** (ver la fila de $P$: ∀ = V implica ∃ = V). Comprobación exhaustiva para todos los predicados posibles sobre $|U|=3$:

In [5]:
U3 = [1, 2, 3]
ok = True
for bits in itertools.product([False, True], repeat=len(U3)):
    Pm = dict(zip(U3, bits))
    f = lambda x, Pm=Pm: Pm[x]
    ok &= (not universal(U3, f)) or existencial(U3, f)
print('∀ ⇒ ∃ en las', 2**len(U3), 'asignaciones posibles:', ok)

∀ ⇒ ∃ en las 8 asignaciones posibles: True


## 3. Existencia única $(\exists! x\in U)\,P(x)$

Según la Definición 2.5:

$$
(\exists! x)P(x)\equiv \underbrace{(\exists x)P(x)}_{\text{Existencia}}\land\underbrace{(\forall y)(\forall z)\,[P(y)\land P(z)\to y=z]}_{\text{Unicidad}}
$$

Se resuelve en dos tablas:
1. **Existencia:** columna de $P$ reducida con $\lor$.
2. **Unicidad:** tabla de pares $(y,z)$ con la columna $P(y)\land P(z)\to y=z$, reducida con $\land$.

Se evalúan tres casos sobre $U=\{1,2,3,4\}$:
- $A(x)$: $x=3$ (un testigo)
- $B(x)$: $x$ es par (dos testigos: 2 y 4)
- $C(x)$: $x>4$ (ningún testigo)

In [6]:
def impl(a, b): return (not a) or b

def tabla_unicidad(U, P):
    filas = []
    for y, z in itertools.product(U, repeat=2):
        py, pz = P(y), P(z)
        filas.append({'y': y, 'z': z, 'P(y)': V(py), 'P(z)': V(pz),
                      'P(y)∧P(z)': V(py and pz), 'y=z': V(y == z),
                      'P(y)∧P(z) → y=z': V(impl(py and pz, y == z))})
    return pd.DataFrame(filas)

def analiza_unico(U, P, titulo):
    print('='*70); print(titulo); print('='*70)
    display(tabla(U, P))
    existe = existencial(U, P)
    tu = tabla_unicidad(U, P)
    unic = all(impl(P(y) and P(z), y == z) for y in U for z in U)
    display(tu)
    viol = tu[tu['P(y)∧P(z) → y=z'] == 'F'][['y', 'z']].values.tolist()
    print('Existencia  (∃x P(x))                    :', V(existe))
    print('Unicidad    (∀y∀z[P(y)∧P(z)→y=z])       :', V(unic), ('  pares que la violan: ' + str(viol)) if viol else '')
    print('∃!x P(x)    = Existencia ∧ Unicidad      :', V(existe and unic))
    assert (existe and unic) == unico(U, P)   # coincide con "exactamente un testigo"

U4 = [1, 2, 3, 4]

In [7]:
analiza_unico(U4, lambda x: x == 3, 'A(x): x = 3   (un testigo)')

A(x): x = 3   (un testigo)


,P(x)
x,
1,F
2,F
3,V
4,F


,y,z,P(y),P(z),P(y)∧P(z),y=z,P(y)∧P(z) → y=z
0,1,1,F,F,F,V,V
1,1,2,F,F,F,F,V
2,1,3,F,V,F,F,V
3,1,4,F,F,F,F,V
4,2,1,F,F,F,F,V
5,2,2,F,F,F,V,V
6,2,3,F,V,F,F,V
7,2,4,F,F,F,F,V
8,3,1,V,F,F,F,V
9,3,2,V,F,F,F,V


Existencia  (∃x P(x))                    : V
Unicidad    (∀y∀z[P(y)∧P(z)→y=z])       : V 
∃!x P(x)    = Existencia ∧ Unicidad      : V


In [8]:
analiza_unico(U4, lambda x: x % 2 == 0, 'B(x): x es par   (dos testigos: falla la unicidad)')

B(x): x es par   (dos testigos: falla la unicidad)


,P(x)
x,
1,F
2,V
3,F
4,V


,y,z,P(y),P(z),P(y)∧P(z),y=z,P(y)∧P(z) → y=z
0,1,1,F,F,F,V,V
1,1,2,F,V,F,F,V
2,1,3,F,F,F,F,V
3,1,4,F,V,F,F,V
4,2,1,V,F,F,F,V
5,2,2,V,V,V,V,V
6,2,3,V,F,F,F,V
7,2,4,V,V,V,F,F
8,3,1,F,F,F,F,V
9,3,2,F,V,F,F,V


Existencia  (∃x P(x))                    : V
Unicidad    (∀y∀z[P(y)∧P(z)→y=z])       : F   pares que la violan: [[2, 4], [4, 2]]
∃!x P(x)    = Existencia ∧ Unicidad      : F


In [9]:
analiza_unico(U4, lambda x: x > 4, 'C(x): x > 4   (sin testigos: falla la existencia)')

C(x): x > 4   (sin testigos: falla la existencia)


,P(x)
x,
1,F
2,F
3,F
4,F


,y,z,P(y),P(z),P(y)∧P(z),y=z,P(y)∧P(z) → y=z
0,1,1,F,F,F,V,V
1,1,2,F,F,F,F,V
2,1,3,F,F,F,F,V
3,1,4,F,F,F,F,V
4,2,1,F,F,F,F,V
5,2,2,F,F,F,V,V
6,2,3,F,F,F,F,V
7,2,4,F,F,F,F,V
8,3,1,F,F,F,F,V
9,3,2,F,F,F,F,V


Existencia  (∃x P(x))                    : F
Unicidad    (∀y∀z[P(y)∧P(z)→y=z])       : V 
∃!x P(x)    = Existencia ∧ Unicidad      : F


In [10]:
pd.DataFrame(
    {'Existencia': ['V', 'V', 'F'], 'Unicidad': ['V', 'F', 'V'], '∃!': ['V', 'F', 'F']},
    index=['A: x=3', 'B: x par', 'C: x>4'])

,Existencia,Unicidad,∃!
A: x=3,V,V,V
B: x par,V,F,F
C: x>4,F,V,F


**Conclusión.** $\exists!$ exige ambas condiciones a la vez. Nótese que en $C$ la unicidad es V *vacuamente* (el antecedente $P(y)\land P(z)$ es siempre F), pero falla la existencia; en $B$ hay existencia y falla la unicidad (par $(2,4)$ y $(4,2)$).

## 4. Negación de proposiciones cuantificadas

Teorema 2.1:
1. $\neg[(\forall x)P(x)]\equiv(\exists x)\neg P(x)$
2. $\neg[(\exists x)P(x)]\equiv(\forall x)\neg P(x)$

Se verifica con una tabla de verdad **exhaustiva**: todas las asignaciones posibles de valores de verdad de $P$ sobre $U=\{u_1,u_2,u_3\}$ ($2^3=8$ filas). Dos expresiones son equivalentes si sus columnas coinciden en todas las filas.

In [11]:
n = 3
filas = []
for bits in itertools.product([True, False], repeat=n):
    p = list(bits)
    filas.append({
        **{f'P(u{i+1})': V(b) for i, b in enumerate(p)},
        '∀xP': V(all(p)),
        '¬∀xP': V(not all(p)),
        '∃x¬P': V(any(not b for b in p)),
        '∃xP': V(any(p)),
        '¬∃xP': V(not any(p)),
        '∀x¬P': V(all(not b for b in p)),
    })
neg = pd.DataFrame(filas)
neg['¬∀ ≡ ∃¬'] = (neg['¬∀xP'] == neg['∃x¬P']).map(V)
neg['¬∃ ≡ ∀¬'] = (neg['¬∃xP'] == neg['∀x¬P']).map(V)
display(neg)
print('Teorema 2.1 (1) verificado en todas las filas:', (neg['¬∀ ≡ ∃¬'] == 'V').all())
print('Teorema 2.1 (2) verificado en todas las filas:', (neg['¬∃ ≡ ∀¬'] == 'V').all())

,P(u1),P(u2),P(u3),∀xP,¬∀xP,∃x¬P,∃xP,¬∃xP,∀x¬P,¬∀ ≡ ∃¬,¬∃ ≡ ∀¬
0,V,V,V,V,F,F,V,F,F,V,V
1,V,V,F,F,V,V,V,F,F,V,V
2,V,F,V,F,V,V,V,F,F,V,V
3,V,F,F,F,V,V,V,F,F,V,V
4,F,V,V,F,V,V,V,F,F,V,V
5,F,V,F,F,V,V,V,F,F,V,V
6,F,F,V,F,V,V,V,F,F,V,V
7,F,F,F,F,V,V,F,V,V,V,V


Teorema 2.1 (1) verificado en todas las filas: True
Teorema 2.1 (2) verificado en todas las filas: True


Las columnas `¬∀xP` y `∃x¬P` son idénticas en las 8 filas, y lo mismo `¬∃xP` y `∀x¬P`: **las dos leyes son tautologías** (en este universo; la demostración general para $|U|=n$ es la ley de De Morgan para $n$ términos).

## 5. Cuantificadores múltiples y orden de cuantificación

Un predicado binario $P(x,y)$ sobre $U$ es una **matriz de verdad** $|U|\times|U|$ (filas $x$, columnas $y$). Cada cuantificador reduce una dimensión:

| Proposición | Cómo se evalúa en la matriz |
|---|---|
| $(\forall x)(\forall y)P$ | todas las celdas V |
| $(\exists x)(\exists y)P$ | alguna celda V |
| $(\forall x)(\exists y)P$ | **cada fila** tiene alguna V |
| $(\exists y)(\forall x)P$ | **alguna columna** es toda V |

### 5.1 Ejemplo del capítulo: $x+y=0$

In [12]:
def matriz(U, P):
    return pd.DataFrame([[V(P(x, y)) for y in U] for x in U],
                        index=pd.Index(U, name='x'), columns=pd.Index(U, name='y'))

def evalua_dobles(U, P):
    M = {(x, y): P(x, y) for x in U for y in U}
    return {
        '∀x∀y': all(M[x, y] for x in U for y in U),
        '∀y∀x': all(M[x, y] for y in U for x in U),
        '∃x∃y': any(M[x, y] for x in U for y in U),
        '∃y∃x': any(M[x, y] for y in U for x in U),
        '∀x∃y': all(any(M[x, y] for y in U) for x in U),
        '∃y∀x': any(all(M[x, y] for x in U) for y in U),
        '∃x∀y': any(all(M[x, y] for y in U) for x in U),
        '∀y∃x': all(any(M[x, y] for x in U) for y in U),
    }

Ur = [-2, -1, 0, 1, 2]
S = lambda x, y: x + y == 0
display(matriz(Ur, S))

# reducción por fila (∃y) y por columna (∀x)
filas_ex = pd.Series({x: V(any(S(x, y) for y in Ur)) for x in Ur}, name='∃y P(x,y)  (por fila)')
cols_all = pd.Series({y: V(all(S(x, y) for x in Ur)) for y in Ur}, name='∀x P(x,y)  (por columna)')
display(filas_ex.to_frame().T); display(cols_all.to_frame().T)

r = evalua_dobles(Ur, S)
print('(∀x)(∃y)(x+y=0) :', V(r['∀x∃y']), ' → cada fila tiene una V (la antidiagonal, y=-x)')
print('(∃y)(∀x)(x+y=0) :', V(r['∃y∀x']), ' → ninguna columna es toda V')

y,-2,-1,0,1,2
x,,,,,
-2,F,F,F,F,V
-1,F,F,F,V,F
0,F,F,V,F,F
1,F,V,F,F,F
2,V,F,F,F,F


,-2,-1,0,1,2
"∃y P(x,y) (por fila)",V,V,V,V,V


,-2,-1,0,1,2
"∀x P(x,y) (por columna)",F,F,F,F,F


(∀x)(∃y)(x+y=0) : V  → cada fila tiene una V (la antidiagonal, y=-x)
(∃y)(∀x)(x+y=0) : F  → ninguna columna es toda V


Se reproduce el ejemplo del texto: $(\forall x)(\exists y)$ es **V** y $(\exists y)(\forall x)$ es **F**; los cuantificadores heterogéneos **no conmutan**.

*(Nota: en $U=\{-2,\dots,2\}$ se tiene $y=-x\in U$ para todo $x$, igual que en $\mathbb R$. Si se usara $U=\{0,1,2\}$, $\forall x\exists y$ sería F porque $-1\notin U$: el resultado depende del universo.)*

### 5.2 Verificación exhaustiva de las leyes

Se generan **todas** las relaciones binarias posibles sobre $U=\{1,2,3\}$ ($2^9=512$ matrices) y se comprueba:

1. $\forall x\forall y\equiv\forall y\forall x$ y $\exists x\exists y\equiv\exists y\exists x$ (homogéneos conmutan).
2. $(\exists y)(\forall x)P\Rightarrow(\forall x)(\exists y)P$ siempre.
3. El recíproco falla en al menos una matriz (contraejemplo).

In [13]:
U3 = [1, 2, 3]
celdas = list(itertools.product(U3, repeat=2))
conmuta_A = conmuta_E = impl_ok = True
contraejemplos = []
n_AE_V = n_EA_V = 0
for bits in itertools.product([False, True], repeat=len(celdas)):
    M = dict(zip(celdas, bits))
    r = evalua_dobles(U3, lambda x, y, M=M: M[x, y])
    conmuta_A &= r['∀x∀y'] == r['∀y∀x']
    conmuta_E &= r['∃x∃y'] == r['∃y∃x']
    impl_ok   &= impl(r['∃y∀x'], r['∀x∃y'])
    if r['∀x∃y'] and not r['∃y∀x']:
        contraejemplos.append(M)
    n_AE_V += r['∀x∃y']; n_EA_V += r['∃y∀x']

print('∀∀ conmutan en las 512 matrices :', conmuta_A)
print('∃∃ conmutan en las 512 matrices :', conmuta_E)
print('(∃y∀x P) ⇒ (∀x∃y P) en las 512  :', impl_ok)
print(f'Matrices con ∀x∃y = V: {n_AE_V};  con ∃y∀x = V: {n_EA_V};  con ∀x∃y=V y ∃y∀x=F (contraejemplos al recíproco): {len(contraejemplos)}')
print('\nUn contraejemplo al recíproco (x filas, y columnas):')
display(matriz(U3, lambda x, y: contraejemplos[0][x, y]))

∀∀ conmutan en las 512 matrices :

 True
∃∃ conmutan en las 512 matrices : True
(∃y∀x P) ⇒ (∀x∃y P) en las 512  : True
Matrices con ∀x∃y = V: 343;  con ∃y∀x = V: 169;  con ∀x∃y=V y ∃y∀x=F (contraejemplos al recíproco): 174

Un contraejemplo al recíproco (x filas, y columnas):


y,1,2,3
x,,,
1,F,F,V
2,F,F,V
3,F,V,F


In [14]:
# Tabla resumen del implicación/recíproco: combinaciones (∃y∀x, ∀x∃y) que ocurren
comb = {}
for bits in itertools.product([False, True], repeat=len(celdas)):
    M = dict(zip(celdas, bits))
    r = evalua_dobles(U3, lambda x, y, M=M: M[x, y])
    k = (V(r['∃y∀x']), V(r['∀x∃y']))
    comb[k] = comb.get(k, 0) + 1
pd.DataFrame([{'∃y∀xP': a, '∀x∃yP': b, 'nº de matrices': c} for (a, b), c in sorted(comb.items(), reverse=True)])

,∃y∀xP,∀x∃yP,nº de matrices
0,V,V,169
1,F,V,174
2,F,F,169


La combinación ($\exists y\forall x=$ V, $\forall x\exists y=$ F) **no aparece**: la implicación es válida. La combinación ($\exists y\forall x=$ F, $\forall x\exists y=$ V) **sí aparece**: el recíproco es falso en general.

## 6. Definiciones con cuantificadores anidados (modelo finito)

Las definiciones del análisis tienen la forma $\forall\varepsilon\,\exists\delta\,\forall x\,[\dots]$. En $\mathbb R$ los universos son infinitos y no hay tabla completa, pero la **estructura** de las reducciones es la misma. Aquí se discretiza con rejillas finitas para ver cómo cada cuantificador reduce una dimensión (∀ = `all`, ∃ = `any`). Es una ilustración, **no** una demostración.

Límite: $\lim_{x\to a}f(x)=L$ con $f(x)=2x$, $a=1$, $L=2$:
$$(\forall\varepsilon>0)(\exists\delta>0)(\forall x)\,[0<|x-a|<\delta\Rightarrow|f(x)-L|<\varepsilon]$$

In [15]:
from fractions import Fraction as Fr

f = lambda x: 2 * x
a, L = Fr(1), Fr(2)
eps_grid   = [Fr(1), Fr(1, 2), Fr(1, 10)]
delta_grid = [Fr(1), Fr(1, 2), Fr(1, 4), Fr(1, 10), Fr(1, 20), Fr(1, 100)]
x_grid     = [Fr(k, 100) for k in range(-100, 301)]          # 0 ... 3 en pasos de 0.01 (incl. negativos)

def cond(eps, delta, x):
    # la implicación interna  0<|x-a|<δ → |f(x)-L|<ε
    return impl(0 < abs(x - a) < delta, abs(f(x) - L) < eps)

# para cada (ε, δ): ∀x [ ... ]
tab = pd.DataFrame(
    [[V(all(cond(e, d, x) for x in x_grid)) for d in delta_grid] for e in eps_grid],
    index=pd.Index([str(float(e)) for e in eps_grid], name='ε'),
    columns=pd.Index([str(float(d)) for d in delta_grid], name='δ'))
display(tab)

# para cada ε: ∃δ [ ∀x ... ]
res = {str(float(e)): V(any(all(cond(e, d, x) for x in x_grid) for d in delta_grid)) for e in eps_grid}
print('∃δ ∀x [...] para cada ε :', res)
print('∀ε ∃δ ∀x [...]          :', V(all(v == 'V' for v in res.values())))

δ,1.0,0.5,0.25,0.1,0.05,0.01
ε,,,,,,
1.0,F,V,V,V,V,V
0.5,F,F,V,V,V,V
0.1,F,F,F,F,V,V


∃δ ∀x [...] para cada ε : {'1.0': 'V', '0.5': 'V', '0.1': 'V'}
∀ε ∃δ ∀x [...]          : V


Cada celda de la tabla es un $\forall x$; luego "$\exists\delta$" = *existe una V en la fila* y "$\forall\varepsilon$" = *todas las filas cumplen eso*. Como $|f(x)-2|=2|x-1|$, basta $\delta\le\varepsilon/2$ y se observa el patrón escalonado (más V hacia la derecha cuando $\delta$ decrece). Además $\delta$ **depende de $\varepsilon$**: el $\delta$ válido cambia por fila.

### Orden de cuantificadores: continuidad ordinaria vs. uniforme

Con $f(x)=x^2$ y $\varepsilon=1$ se compara
- **Ordinaria:** $\forall x_0\,\exists\delta\,\forall x$ ($\delta$ puede depender de $x_0$).
- **Uniforme:** $\exists\delta\,\forall x_0\,\forall x$ ($\delta$ sólo depende de $\varepsilon$).

En la tabla, cada fila es un punto $x_0$ y cada columna un $\delta$; la celda dice si ese $\delta$ sirve en ese punto. Ordinaria = *cada fila tiene alguna V*; uniforme = *alguna columna es toda V*: exactamente la estructura de la sección 5.

In [16]:
g = lambda x: x * x
eps = Fr(1)
deltas = [Fr(1), Fr(1, 2), Fr(1, 4), Fr(1, 10), Fr(1, 20), Fr(1, 50), Fr(1, 100), Fr(1, 1000)]

def sirve(x0, d):
    # malla fina de x en [x0-1, x0+1] (paso 1/2000); basta porque δ ≤ 1
    x0 = Fr(x0)
    xs = [x0 + Fr(k, 2000) for k in range(-2000, 2001)]
    return all(impl(abs(x - x0) < d, abs(g(x) - g(x0)) < eps) for x in xs)

def tabla_cont(puntos):
    return pd.DataFrame([[V(sirve(x0, d)) for d in deltas] for x0 in puntos],
                        index=pd.Index(puntos, name='x0'),
                        columns=pd.Index([str(float(d)) for d in deltas], name='δ'))

def veredicto(M):
    ordi = all((M.loc[i] == 'V').any() for i in M.index)
    uni  = any((M[c] == 'V').all() for c in M.columns)
    print('Ordinaria ∀x0 ∃δ (cada fila con alguna V):', V(ordi), '| Uniforme ∃δ ∀x0 (alguna columna toda V):', V(uni))

M = tabla_cont([0, 1, 2, 5, 10, 50, 100])
display(M); veredicto(M)

δ,1.0,0.5,0.25,0.1,0.05,0.02,0.01,0.001
x0,,,,,,,,
0,V,V,V,V,V,V,V,V
1,F,F,V,V,V,V,V,V
2,F,F,F,V,V,V,V,V
5,F,F,F,F,V,V,V,V
10,F,F,F,F,V,V,V,V
50,F,F,F,F,F,F,V,V
100,F,F,F,F,F,F,F,V


Ordinaria ∀x0 ∃δ (cada fila con alguna V): V | Uniforme ∃δ ∀x0 (alguna columna toda V): V


Lo que muestra la tabla:
- Cada fila tiene alguna V: $x^2$ es continua en cada punto (**ordinaria = V**).
- El $\delta$ que sirve **se encoge al crecer $x_0$**: $\delta$ depende de $x_0$.
- La columna $\delta=0.001$ es toda V **porque el conjunto de puntos $x_0$ es acotado** ($x_0\le 100$): en un conjunto acotado siempre existe un $\delta$ uniforme. Para romper la uniformidad hace falta un conjunto **no acotado** ($I=[0,\infty)$): entonces $\delta\approx\varepsilon/(2x_0)\to0$ y ningún $\delta$ fijo sirve. Una tabla finita sólo puede *sugerirlo*; lo que sí se puede tabular es el mayor $\delta$ admisible en cada $x_0$:

In [17]:
import math
# mayor δ que sirve en x0 (lado derecho, el más restrictivo): x^2 - x0^2 < 1  ⇒  δ* = sqrt(x0^2+1) - x0
dstar = pd.DataFrame({'x0': [0, 1, 10, 100, 1000, 10**4, 10**6]})
dstar['δ*(x0)'] = dstar['x0'].map(lambda x0: math.sqrt(x0**2 + 1) - x0)
dstar['¿sirve δ=0.001?'] = (dstar['δ*(x0)'] > 0.001).map(V)
display(dstar)

,x0,δ*(x0),¿sirve δ=0.001?
0,0,1.000000e+00,V
1,1,4.142136e-01,V
2,10,4.987562e-02,V
3,100,4.999875e-03,V
4,1000,4.999999e-04,F
5,10000,5.000000e-05,F
6,1000000,5.000038e-07,F


El $\delta$ máximo admisible $\delta^*(x_0)=\sqrt{x_0^2+1}-x_0\approx 1/(2x_0)$ tiende a $0$: para **cualquier** $\delta>0$ fijo existe un $x_0$ (en $[0,\infty)$) donde ya no sirve, así que la columna "toda V" no existe para el universo completo: $(\exists\delta)(\forall x_0)$ es **F** mientras $(\forall x_0)(\exists\delta)$ es **V** (cada $x_0$ tiene su $\delta^*>0$). Es la versión "tabla" de $(\forall x_0)(\exists\delta)\not\Rightarrow(\exists\delta)(\forall x_0)$, como en la sección 5.

> **Advertencia.** Con mallas finitas las conclusiones dependen de la rejilla; la tabla ilustra la estructura lógica, no sustituye la demostración analítica.

## Resumen

| Cuantificador | Tabla de verdad sobre $U$ finito | V si… | Contraejemplo / testigo |
|---|---|---|---|
| $\forall x\,P(x)$ | conjunción de la columna $P$ | todas las filas V | contraejemplo = fila F |
| $\exists x\,P(x)$ | disyunción de la columna $P$ | alguna fila V | testigo = fila V |
| $\exists!x\,P(x)$ | (∃) ∧ tabla de pares $P(y)\land P(z)\to y=z$ | exactamente una fila V | dos testigos distintos la refutan |
| $\neg\forall\equiv\exists\neg$, $\neg\exists\equiv\forall\neg$ | columnas idénticas en las $2^n$ filas | tautologías | — |
| $\forall\forall$, $\exists\exists$ | matriz: todas / alguna celda | conmutan | — |
| $\exists\forall\Rightarrow\forall\exists$ | matriz: columna toda V ⇒ cada fila con V | implicación válida; recíproco no | $x+y=0$ |